In [7]:
import pyspark
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('RddAssignments').getOrCreate()

In [8]:
rdd =spark.sparkContext.textFile("/home/sois/sample2.txt")
rdd.collect()

['HII VIKRAM', 'HELLO ADITYA']

In [9]:
inputs=['hello world','hello hello world']
Rdd1Way = spark.sparkContext.parallelize(inputs)
Rdd1Way.collect()

['hello world', 'hello hello world']

In [19]:
rdd.flatMap(lambda d: d.split(' ')).map(lambda d:(d,1)).reduceByKey(lambda a,b : a+b).collect()

[('HII', 1), ('VIKRAM', 1), ('HELLO', 1), ('ADITYA', 1)]

In [12]:
lowerCase = rdd.flatMap(lambda x:x.split()).map(lambda x:x.lower())
lowerCase.collect()

['hii', 'vikram', 'hello', 'aditya']

In [13]:
upperCase =  rdd.flatMap(lambda x:x.split()).map(lambda x:x.upper())
upperCase.collect()


['HII', 'VIKRAM', 'HELLO', 'ADITYA']

In [14]:
firstLetterCapital = rdd.flatMap(lambda x:x.split()).map(lambda x:x.capitalize())
firstLetterCapital.collect()

['Hii', 'Vikram', 'Hello', 'Aditya']

In [15]:
wordCounts = (rdd.flatMap(lambda x: x.lower().split()).map(lambda x: (x, 1)).reduceByKey(lambda a, b: a + b))      
wordCounts.collect()

[('hii', 1), ('vikram', 1), ('hello', 1), ('aditya', 1)]

In [16]:
word = "hii"
filtered = rdd.filter(lambda x: word.lower() in x.lower())
filtered.collect()

['HII VIKRAM']

In [17]:
longestLength = (rdd.flatMap(lambda x: x.split()).map(lambda x: len(x)))
maxLength = max(longestLength.collect())
print(maxLength)

6


In [26]:
mapped_rdd = spark.sparkContext.parallelize(["58001","57023","38045","39012","47056"]) \
    .map(lambda x: int(x)) \
    .map(lambda x: (x,
        "BDA" if 58000 <= x < 59000 else
        "AIML" if 57000 <= x < 58000 else
        "VLSI" if 38000 <= x < 39000 else
        "ES" if 39000 <= x < 40000 else
        "CDC" if 47000 <= x < 48000 else "Unknown"))

mapped_rdd.collect()

[(58001, 'BDA'),
 (57023, 'AIML'),
 (38045, 'VLSI'),
 (39012, 'ES'),
 (47056, 'CDC')]

In [30]:
rdd1 = spark.sparkContext.textFile("/home/sois/sample3.txt")
rdd1.collect()

['10 20 30', '5 15', '40']

In [35]:
numbers = rdd1.flatMap(lambda x: x.split()).map(int)

print("Max:", numbers.max())
print("Min:", numbers.min())
print("Sum:", numbers.sum())
print("Mean:", numbers.sum() / numbers.count())

Max: 40
Min: 5
Sum: 120
Mean: 20.0


In [55]:
citizen_rdd = spark.sparkContext.textFile("/home/sois/citizen.txt")
state_rdd = spark.sparkContext.textFile("/home/sois/state_mapping.txt")
citizen_rdd.collect()
state_rdd.collect()

['Ravi,1999-05-10,9876543210,ravi@gmail.com,Karnataka',
 'Anu,2000-01-15,9123456780,anu@gmail.com,TamilNadu']

In [51]:
state_map = dict(state_rdd
                 .map(lambda x: x.split(","))
                 .map(lambda x: (x[0], x[1]))
                 .collect())

In [52]:
compressed = citizen_rdd.map(lambda l: l.split(",")) \
    .map(lambda x: x[:-1] + [state_map.get(x[-1], x[-1])]) \
    .map(lambda x: ",".join(x))

compressed.collect()

['Ravi,1999-05-10,9876543210,ravi@gmail.com,Karnataka',
 'Anu,2000-01-15,9123456780,anu@gmail.com,TamilNadu']

In [60]:
rdd3 = spark.sparkContext.textFile("/home/sois/students.txt")
rdd3.collect()

['Ravi,MIT,CSE,Boy',
 'Anu,IIT,AI,Girl',
 'Rahul,MIT,ECE,Boy',
 'Sneha,IIT,CSE,Girl',
 'Kiran,BITS,AI,Boy',
 'Meena,MIT,AI,Girl',
 'Arjun,BITS,CSE,Boy',
 'Divya,IIT,ECE,Girl']

In [67]:
students = rdd3.map(lambda x: x.split(","))
students.collect()

[['Ravi', 'MIT', 'CSE', 'Boy'],
 ['Anu', 'IIT', 'AI', 'Girl'],
 ['Rahul', 'MIT', 'ECE', 'Boy'],
 ['Sneha', 'IIT', 'CSE', 'Girl'],
 ['Kiran', 'BITS', 'AI', 'Boy'],
 ['Meena', 'MIT', 'AI', 'Girl'],
 ['Arjun', 'BITS', 'CSE', 'Boy'],
 ['Divya', 'IIT', 'ECE', 'Girl']]

In [68]:
institute_count = students \
    .map(lambda x: (x[1], 1)) \
    .reduceByKey(lambda a, b: a + b)

institute_count.collect()

[('MIT', 3), ('IIT', 3), ('BITS', 2)]

In [69]:
program_count = students \
    .map(lambda x: (x[2], 1)) \
    .reduceByKey(lambda a, b: a + b)

program_count.collect()

[('CSE', 3), ('AI', 3), ('ECE', 2)]

In [70]:
selected = students.filter(lambda x: x[1] == "MIT")

gender_selected = selected \
    .map(lambda x: (x[3], 1)) \
    .reduceByKey(lambda a, b: a + b)

gender_selected.collect()

[('Boy', 2), ('Girl', 1)]

In [71]:
rdd4= spark.sparkContext.textFile("/home/sois/Temperature.txt")
rdd4.collect()

['1995-01-01,24.5,Bangalore,India,12.97,77.59']

In [76]:
data = rdd4.map(lambda x: x.split(","))
data.collect()

[['1995-01-01', '24.5', 'Bangalore', 'India', '12.97', '77.59']]

In [79]:
records = data.map(lambda x: (x[2], float(x[1])))  
temps = records.map(lambda x: x[1])

print("Maximum temp", temps.max())
print("Minimum temp:", temps.min())

Maximum temp 24.5
Minimum temp: 24.5


In [81]:
city_count = records.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b)
city_count.collect()

[('Bangalore', 1)]

In [89]:
bangalore = records.filter(lambda x: x[0] == "Bangalore").map(lambda x: x[1])

print("Bangalore Max:", bangalore.max())
print("Bangalore Min:", bangalore.min())

Bangalore Max: 24.5
Bangalore Min: 24.5


In [94]:
rdd4 = spark.sparkContext.textFile("/home/sois/bank.txt")
data = rdd4.map(lambda x: x.split(","))
data.collect()

[['B001', '1001', '12-01-2023', 'credit', '5000'],
 ['B001', '1002', '15-03-2023', 'debit', '2000'],
 ['B002', '1003', '10-05-2022', 'credit', '7000'],
 ['B001', '1001', '20-06-2023', 'debit', '1000'],
 ['B002', '1004', '11-07-2022', 'credit', '3000'],
 ['B003', '1005', '01-01-2023', 'debit', '4000'],
 ['B002', '1003', '21-09-2023', 'credit', '6000']]

In [97]:
unique_customers = data.map(lambda x: x[1]).distinct().count()
print("Unique Customers:", unique_customers)

Unique Customers: 5


In [98]:
unique_banks = data.map(lambda x: x[0]).distinct().count()
print("Unique Banks:", unique_banks)

Unique Banks: 3


In [99]:
customers_per_bank = data .map(lambda x: (x[0], x[1])).distinct().map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b)
customers_per_bank.collect()

[('B001', 2), ('B002', 2), ('B003', 1)]

In [100]:
account_number = "1001"
transaction_count = data.filter(lambda x: x[1] == account_number).count()
print("Transactions Account:", transaction_count)

Transactions for Account: 2
